In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()
llm = ChatOpenAI(proxy_model_name='gpt-4o')

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer=InMemorySaver()
workflow=graph.compile(checkpointer=checkpointer)

In [10]:
config1={"configurable":{"thread_id":"1"}}
workflow.invoke({'topic':'pizza'},config=config1)

{'topic': 'pizza',
 'joke': "Why did the pizza go to the party by itself?\n\nBecause it couldn't find its crustworthy companion!",
 'explanation': 'The joke, "Why did the pizza go to the party by itself? Because it couldn\'t find its crustworthy companion!" is a classic example of humor that hinges on wordplay and personification.\n\n1. **Wordplay:**\n   - **"Crustworthy"**: This is a pun on the word "trustworthy." The joke creates a play on words by combining "crust," which is a part of pizza, with "worthy," to mimic "trustworthy." It humorously suggests that the pizza needs a companion that it can "trust" or rely on, but specifically one connected to pizza, hence the use of "crust." The neologism engages the audience by cleverly tying the concept of reliability to something directly related to pizzas, i.e., the crust.\n\n2. **Personification:**\n   - Here, the pizza is anthropomorphized, given human-like qualities and actions, such as attending a party and seeking companionship. This